In [1]:
import pathlib, sys
import matplotlib
matplotlib.use('Agg')
_root = pathlib.Path.cwd()
if not (_root / 'src').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# 05 · Arena: Retrieval

Retrieval splits the cleaning story in two. **BM25** matches on surface tokens, so normalization that increases term overlap — stemming `running`/`runs`/`run` together — tends to help it. **Dense** retrieval encodes the raw text with a transformer, so the same stemming feeds the encoder mangled tokens and hurts. The right cleaning depends on the retriever.

In [2]:
import numpy as np
import pandas as pd
from src.cleaning.pipelines import PIPELINE_NAMES, clean
from src.datasets.corpora import load_retrieval
from src.evaluation.retrieval import evaluate_retrieval, ndcg_at_k, recall_at_k
from src.models.bm25 import BM25Retriever
from src.models.transformer import TransformerEmbedder
from src.visualisation import plot_pipeline_scores, save_figure, save_json
corpus = load_retrieval()
embedder = TransformerEmbedder()
K = 5

## A dense retriever on transformer embeddings

Encode the cleaned documents and queries, rank by cosine similarity, and score the same nDCG / recall BM25 gets.

In [3]:
def dense_eval(corpus, pipeline, k=K):
    docs = embedder.encode(corpus.documents, pipeline=pipeline)
    qs = embedder.encode(corpus.queries, pipeline=pipeline)
    docs = docs / (np.linalg.norm(docs, axis=1, keepdims=True) + 1e-9)
    qs = qs / (np.linalg.norm(qs, axis=1, keepdims=True) + 1e-9)
    sims = qs @ docs.T
    ndcgs, recalls = [], []
    for qi in range(len(corpus.queries)):
        ranked = [int(i) for i in np.argsort(-sims[qi])[:k]]
        rel = corpus.qrels.get(qi, set())
        ndcgs.append(ndcg_at_k(ranked, rel, k))
        recalls.append(recall_at_k(ranked, rel, k))
    return float(np.mean(ndcgs)), float(np.mean(recalls))

In [4]:
rows = []
for p in PIPELINE_NAMES:
    bm = evaluate_retrieval(BM25Retriever(), corpus, clean_fn=lambda t, _p=p: clean(t, _p), k=K)
    rows.append({'model': 'bm25', 'pipeline': p, 'ndcg': bm.ndcg, 'recall': bm.recall})
    ndcg, recall = dense_eval(corpus, p)
    rows.append({'model': 'dense', 'pipeline': p, 'ndcg': ndcg, 'recall': recall})
retrieval = pd.DataFrame(rows)
save_json(retrieval, 'nb05_retrieval_arena.json')
retrieval.round(3)

,model,pipeline,ndcg,recall
0,bm25,none,0.186,0.3
1,dense,none,1.000,1.0
2,bm25,minimal,0.186,0.3
3,dense,minimal,1.000,1.0
4,bm25,standard,0.632,0.9
5,dense,standard,1.000,1.0
6,bm25,aggressive,0.984,1.0
7,dense,aggressive,1.000,1.0


## nDCG by pipeline and retriever

In [5]:
fig = plot_pipeline_scores(retrieval, metric='ndcg')
fig.axes[0].set_ylabel('nDCG@%d' % K)
fig.axes[0].set_title('Retrieval nDCG by pipeline: BM25 vs dense')
save_figure(fig, 'nb05_retrieval_ndcg.png')

In [6]:
fig = plot_pipeline_scores(retrieval, metric='recall')
fig.axes[0].set_ylabel('recall@%d' % K)
fig.axes[0].set_title('Retrieval recall by pipeline: BM25 vs dense')
save_figure(fig, 'nb05_retrieval_recall.png')

## The effect of aggressive stemming, per retriever

In [7]:
pivot = retrieval.pivot_table(index='model', columns='pipeline', values='ndcg')
effect = pd.DataFrame({
    'model': pivot.index,
    'raw(none)': pivot['none'].values,
    'aggressive': pivot['aggressive'].values,
    'aggressive_minus_raw': (pivot['aggressive'] - pivot['none']).values,
}).reset_index(drop=True)
save_json(effect, 'nb05_stemming_effect.json')
effect.round(3)

,model,raw(none),aggressive,aggressive_minus_raw
0,bm25,0.186,0.984,0.798
1,dense,1.000,1.000,0.000


## Takeaway

The general rule the arena surfaces: lexical methods like BM25 benefit from normalization that increases term overlap, while dense methods want the raw text their encoder expects. Aggressive stemming is a reasonable lever for a sparse retriever and a liability for a dense one — so, as with classification, the correct cleaning depends on the model, not on habit.